__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [1]:
import numpy as np
import pandas as pd
# from typing import override, overload
import re
import string
from utils.constants import chat_words
import os

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (pack_padded_sequence, pad_packed_sequence)

#### Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


#### Preprocessing

In [ ]:
# DATAPATH = 'data/spam.csv'

# dataset_v1 = pd.read_csv(
#     DATAPATH,
#     header=0,
#     usecols=[0, 1],
#     names=["label", "message"],
#     encoding="utf-8",
#     encoding_errors="ignore"
# )

# df = dataset_v1.copy()
# df = df.dropna(subset=["label", "message"]).copy()
# df["label"] = df["label"].astype(str).str.strip().str.lower()
# df["message"] = df["message"].astype(str)

# print(f"Messages: {len(df):,}")
# print("Label counts:", df["label"].value_counts().to_dict())
# df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
1041,ham,I'm in class. Will holla later
5488,ham,K. I will sent it again
1365,spam,HOT LIVE FANTASIES call now 08707509020 Just 2...
2516,ham,Yes.i'm in office da:)
1355,ham,Convey my regards to him


In [4]:
def remove_number_middle_of_text(text):
    pattern = re.compile(r'[a-zA-Z]+')
    text = pattern.sub(r"", text)
    return " ".join(text)

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words.keys():
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
        
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text

In [5]:
# text_preprocessing("Yes da. Any plm at ur office")

text = "Yes da. Any plm at ur office"
text = remove_htmltags(text)
text = remove_urls(text)
text = remove_punctuations(text)
text = eloborate_chat_conversion(text)
text = corrected_spelling(text)
text = remove_stopwords(text)
text = word_tokenize(text)
text = lemmatize_words(text)
text

['yes', 'problem', 'office']

In [ ]:
# # text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
# # https://www.helo_texts.com
# # http://localhost:5000"""+df['message'][0])
# df['tokens'] = df['message'].apply(text_preprocessing)
# corpus_tokens = df['tokens'].tolist()

#### Remove the empty tokens

In [ ]:
# null_indices = df[df['tokens'].apply(lambda x: len(x) == 0)].index
# df.drop(null_indices, inplace=True)

In [ ]:
# ' '.join(df['tokens'].iloc[0])

'go juror point crazy available bugi n great world la e buffet cine get amores wat'

#### For deleting the memory using gc

In [6]:
# import gc

# # del X_train
# gc.collect()

In [ ]:
# df['cleaned_text'] = df['tokens'].apply(' '.join)
# DATA_DIR = os.path.abspath("./data")
# df.to_csv(f"{DATA_DIR}/processed.csv", index=True)

#### Processed data

Need to fix the cleaned text

In [6]:
df = pd.read_csv(os.path.abspath("./data/processed.csv"))
df.sample(5)


,Unnamed: 0,label,message,tokens,cleaned_text
4006,4012,ham,Ok.,['ok'],ok
519,521,ham,Shall i come to get pickle,"['shall', 'come', 'get', 'pickle']",shall come get pickle
3436,3442,ham,Your board is working fine. The issue of overh...,"['board', 'work', 'fine', 'issue', 'overheat',...",board work fine issue overheat also relove sti...
3055,3060,ham,What u talking bout early morning? It's almost...,"['talk', 'bout', 'early', 'morning', 'almost',...",talk bout early morning almost noon
2395,2399,ham,Oh... Kay... On sat right?,"['oh', 'kay', 'sit', 'right']",oh kay sit right


### Features and targets

In [7]:
X = df['cleaned_text']
y = df['label']

#### Train, validation and Test data

In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp, shuffle=True)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print('\n')
print(f'X_val: {X_val.shape}')
print(f'y_val: {y_val.shape}')
print('\n')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (3893,)
y_train: (3893,)


X_val: (834,)
y_val: (834,)


X_test: (835,)
y_test: (835,)


#### Convert target to numerical

In [9]:
le = LabelEncoder()
le.classes_ = np.array(['spam', 'ham'])

y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)
y_test_num = le.transform(y_test)

In [10]:
df['label'].value_counts()

label
ham     4815
spam     747
Name: count, dtype: int64

## Train

#### Build vocabulary

In [13]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
UNK_IDX = 1

vocab = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}

for sentence in X_train.fillna("").tolist():
    for token in str(sentence).split():
        if token not in vocab:
            vocab[token] = len(vocab)

In [ ]:
# print("Vocabulary size:", len(vocab))

Vocabulary size: 5851


In [54]:
def text_to_indices(sentence, vocab):
    numerical_sentences = []
    for token in str(sentence).split():
        token_id = vocab.get(token, UNK_IDX)
        numerical_sentences.append(token_id)
    return numerical_sentences

def convert_to_numerical_sentence(texts: list|str):
    print(f"convert_to_numerical_sentence:texts {texts}")
    X_train_numerical_sentences = []

    if isinstance(texts, str):
        texts_indices = text_to_indices(texts, vocab)
        return texts_indices
    else:
        for sentence in texts:
            texts_indices = text_to_indices(sentence, vocab)
            X_train_numerical_sentences.append(texts_indices)
        return X_train_numerical_sentences

In [15]:
X_train_numerical_sentences = convert_to_numerical_sentence(X_train.fillna("").tolist())
X_train_numerical_sentences[:3]

[[2, 3, 4, 5, 6, 7], [8, 9, 10, 11, 12, 13, 14, 10], [15, 16, 17, 18, 19]]

#### Padding

In [ ]:
TRAIN_SEQ_MAX_LEN = max([ len(seq) for seq in X_train_numerical_sentences ])

def padded_numerical_sentences(numeric_sentences: list[list] | list):
    # Empty input: same behavior as before (returns an empty list)
    if len(numeric_sentences) == 0:
        return []

    # 1D input like [1, 2, 3] -> treat as one sequence
    is_1d = not isinstance(numeric_sentences[0], (list, tuple))
    sequences = [numeric_sentences] if is_1d else numeric_sentences

    padded_seq = []
    for seq in sequences:
        seq = list(seq)[:TRAIN_SEQ_MAX_LEN]
        if len(seq) == 0:
            seq = [UNK_IDX]

        pad_seq = seq + ([PAD_IDX] * (TRAIN_SEQ_MAX_LEN - len(seq)))
        padded_seq.append(pad_seq)

    # 1D in -> 1D out, 2D in -> 2D out
    return padded_seq[0] if is_1d else padded_seq
    

X_train_padded_seqs = padded_numerical_sentences(X_train_numerical_sentences)

In [71]:
len(X_train_padded_seqs)

3893

### Preserving the actual sequence length

In [27]:
X_train_lengths = []
for seq in X_train_numerical_sentences:
    X_train_lengths.append(min(len(seq), TRAIN_SEQ_MAX_LEN))

# handle empty sequences
X_train_lengths = [max(length, 1) for length in X_train_lengths]

In [28]:
len(X_train_padded_seqs)

3893

#### Dataset and Dataloader

In [29]:

# class SentimentDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = torch.tensor(y, dtype=torch.long)
        
#     def __len__(self):
#         return len(self.X)
    
#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]

class SentimentDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        # For BCEWithLogitsLoss (binary classification)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return (self.X[idx], self.lengths[idx], self.y[idx])



#### Prepare Sequence

### Word2Vec

In [ ]:
from gensim.models import KeyedVectors
import gensim.downloader as api
word2vec_model = api.load('word2vec-google-news-300')
# # wv_model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True, limit=500000)

In [ ]:
# def text_to_vector(texts, wv=word2vec_model, dim=300):
#     vectors = []
#     for text in texts:
#         tokens = text.split()
#         token_vectors = []
#         for token in tokens:
#             if token in wv:
#                 token_vectors.append(wv[token])
#             else:

In [24]:
# text_to_vector(X_train.tolist()[1])

In [ ]:
# word2vec_model['hello'].shape

(300,)

In [30]:
train_dataset = SentimentDataset(X_train_padded_seqs, X_train_lengths, y_train_num)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for token_ids, lengths, labels in train_dataloader:

    print("Token IDs:", token_ids.shape)
    print("Lengths:", lengths.shape)
    print("Labels:", labels.shape)

    print("Token dtype:", token_ids.dtype)
    print("Length dtype:", lengths.dtype)
    print("Label dtype:", labels.dtype)

    break

Token IDs: torch.Size([32, 79])
Lengths: torch.Size([32])
Labels: torch.Size([32])
Token dtype: torch.int64
Length dtype: torch.int64
Label dtype: torch.float32


In [31]:
x = torch.tensor([1, 2, 3, 5, 6], dtype=torch.long)

for token_ids, lengths, labels in train_dataloader:
    x = token_ids
    x_range = torch.arange(x.size(1), device=device).unsqueeze(0)
    mask = x_range < lengths.to(device).unsqueeze(1)
    mask = mask.unsqueeze(1)
    print(x.shape)
    break

torch.Size([32, 79])


## Model

In [32]:
class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, dense_dim=128, dropout=0.3):
        super().__init__()
        
        # 1. embeddings
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        
        # 2. Bidirectional LSTM
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        
        # BiLSTM produces hidden_dim * 2 features
        lstm_output_dim = hidden_dim * 2    
        
        # 3. Dropout
        self.dropout1 = nn.Dropout(dropout)
        
        # 4. Dense layer
        self.fc1 = nn.Linear(in_features=lstm_output_dim, out_features=dense_dim)
        self.relu = nn.ReLU()
        
        # 5. Additional dropout
        self.dropout2 = nn.Dropout(dropout)
        
        # 6. Output layer
        self.fc2 = nn.Linear(in_features=dense_dim, out_features=1)
        
    # @override
    def forward(self, X, lengths):
        lengths_cpu = lengths.cpu()
        lengths_dev = lengths.to(device)
        # input - [batch, sequence]
        X = self.embedding(X) 
        
        # output - [batch, sequence, embedding_dim]
        # Pack sequences to exclude padding from LSTM processing
        packed = pack_padded_sequence(X, lengths=lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        
        # include padding after process
        # output - [batch, sequence, hidden_dim * 2]
        lstm_output, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=X.size(1))
        
        # create a mask for actual tokens
        mask = (
            torch.arange(X.size(1), device=X.device).unsqueeze(0) < lengths_dev.unsqueeze(1)
        )
        mask = mask.unsqueeze(-1)
        
        # masked mean pooling
        masked_output = lstm_output * mask
        
        pooled = masked_output.sum(dim=1) / (lengths.to(X.device).unsqueeze(1))
        
        # output: [batch, hidden_dim * 2]
        
        # Dense transformation
        X = self.dropout1(pooled)
        
        X = self.fc1(X)
        X = self.relu(X)
        X = self.dropout2(X)
        
        # output logits: [batch, 1]
        logits = self.fc2(X)
        
        return logits.squeeze(-1)


In [33]:
len(vocab)

5851

In [34]:
model = SentimentNet(vocab_size=len(vocab)).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(params=model.parameters(), lr=0.001)

### Training Loop

In [ ]:

epochs = 100

for epoch in range(epochs):
    
    model.train()
    total_loss = 0.0
    
    for token_ids, lengths, labels in train_dataloader:
        token_ids = token_ids.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        
        # forward pass
        logits = model(token_ids, lengths)
        
        # calculate loss
        loss = criterion(logits, labels)
        
        # backpropagation
        loss.backward()
        
        # update parameters
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 10 == 0:
        print(f'Epoch: {epoch}, Loss: {total_loss / len(train_dataloader):.4f}')



#### Prediction function

In [75]:
# prediction function

def predict_sentiment(text, model, vocab, device, max_len=200):
    model.eval()

    cleaned_texts = ' '.join(text_preprocessing(text))

    numeric_sentence = convert_to_numerical_sentence(cleaned_texts)
    padded_numeric_sentence = padded_numerical_sentences(numeric_sentence)

    current_seq_length = len(numeric_sentence)

    input_tensor = torch.tensor(
        [padded_numeric_sentence],
        dtype=torch.long
    ).to(device)

    print(f'input_tensor: {input_tensor}')

    length_tensor = torch.tensor([current_seq_length], dtype=torch.long)

    with torch.no_grad():
        logits = model(input_tensor, length_tensor)
        probability = torch.sigmoid(logits).item()

    prediction = "Positive" if probability >= 0.5 else "Negative"

    return {
        "text": text,
        "positive_probability": probability,
        "negative_probability": 1 - probability,
        "prediction": prediction
    }
    
predict_sentiment(
    text="The movie is not so much good.",
    model=model,
    vocab=vocab,
    device=device,
    max_len=79
)

convert_to_numerical_sentence:texts movie much good
input_tensor: tensor([[689,  27, 212,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0]], device='cuda:0')


{'text': 'The movie is not so much good.',
 'positive_probability': 3.987827845719494e-09,
 'negative_probability': 0.9999999960121722,
 'prediction': 'Negative'}

#### Test with validation

In [ ]:

model.eval()
with torch.no_grad():
    logits = model(token_ids)


### Debug

In [72]:
text = "The movie is not so much good."
cleaned_text = " ".join(text_preprocessing(text))
numeric_sentence = convert_to_numerical_sentence(cleaned_text)
padded_sentence = padded_numerical_sentences(numeric_sentence)
padded_sentence

convert_to_numerical_sentence:texts movie much good


[689,
 27,
 212,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [14]:
import numpy as np


np.unique(y_train_num, return_counts=True)

(array([0, 1]), array([3370,  523]))

In [17]:
y_train.value_counts()

label
ham     3370
spam     523
Name: count, dtype: int64